# MDR-TS v9.1
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Wed Feb 11th

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v9.1
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/base/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

_Yoinked form `domain_analysis/02_soil_moisture.ipynb`_

### Gated Model (Wetness-Based Regimes)

- One global regressor is getting forced to “average” across incompatible **dynamical regimes**.
- Periods with high rainfall influence, fast soil moisture response, and elevated variance behave fundamentally differently from stable conditions.
- Stable (low-variance) regimes show strong, learnable structure and should not be dragged down by high-variance events.

So the gate is basically:  

**“What hydrological regime am I in?” → pick the right predictor.**

### Core design

Two experts + a gate (mixture-of-experts vibe)

- `Expert A` (stable-regime expert): trained on low-variance, regular conditions where temporal structure is strong
- `Expert B` (dynamic-regime expert): trained on high-variance, event-driven conditions (rain-driven anomalies)

Then:
- Gate outputs a weight `g(x)` in `[0, 1]`
- Final prediction:
  - `yhat = g(x)*yhat_dynamic + (1-g(x))*yhat_stable`

Key idea: even if the dynamic expert is mediocre, the gate can lean away from it except when strong wetness signals are present.

### 1. What does the gate see (features)

Keep it stupid-simple at first. The gate should only use signals that cleanly separate **stable vs dynamic hydrological behavior**.

**Candidates (as of right now)**  
- `G_rain_sum_7d` or `G_rain_sum_30d`
- `G_API`
- `precip_mm`
- optionally `DOY` (context, not the primary separator)
- optionally `LST_modis` (temperature proxy for evap / snowmelt influence)

The gate should **not** see hundreds of features. If it does, it will just become another overfit regressor.

### 2. Define “wet / dynamic” in a defensible way

Instead of season bins, regimes are defined **relative to rainfall intensity or persistence**.

**Quantile-based logic (preferred)**
- dynamic (wet) if `GATE_COL >= quantile(train, q)`
- stable (dry) otherwise

This makes the gate:
- adaptive across stations
- robust to seasonal and spatial differences
- focused on *relative* regime shifts, not calendar labels

Could also be soft:
- gate learns a smooth transition between stable and dynamic regimes

### 3. Pick expert models

#### `Expert A` (stable-regime expert)

The main workhorse:
- tuned XGB (or stacked XGB + RF)
- uses the full, validated temporal feature set
- optimized for consistency and structure

#### `Expert B` (dynamic-regime expert)

Focused on anomalies:
- can use the same model class or a simpler one
- benefits from rain impulse features and short-horizon dynamics
- goal is not heroics, but capturing event-driven deviations better than Expert A

### 4. Train plan (the clean, non-leaky way)

> Note to self: DO NOT train the gate on test data.

A safe approach:
1. Train `Expert A` on **stable-regime samples** from the train split only
2. Train `Expert B` on **dynamic-regime samples** from the train split only
3. Define gate thresholds using **train split statistics only**
4. Apply gate to val and test for routing and evaluation

Optional extension:
- train a soft gate using train-only data to blend experts smoothly

### 5. Gate variants (to try)

**Variant 1: Hard wetness gate**
- If dynamic → use Expert B
- Else → use Expert A

**Variant 2: Soft wetness gate (recommended)**
- `g = sigmoid(w0 + w1*rain_sum + w2*API + w3*precip)`
- final prediction blends experts using `g`

**Variant 3: Performance-based gate (advanced)**
- gate learns which expert performs better using CV residuals
- predicts expert preference rather than physical regime directly

### 6. How to evaluate

Report:
- Overall test $R^2$
- $R^2$ by wet vs stable regime
- $R^2$ by season (diagnostic, not primary)
- worst-bin performance (high-variance tails)
- optional: calibration and error distributions

**Success criteria**
- Does not need to dominate everywhere
- Should:
  - improve overall stability
  - preserve or improve stable-regime performance
  - reduce collapse during high-variance events

> Note to self: Gate must be trained carefully or it will learn station quirks.  
> Note to self: Optical features are unstable during dynamic periods; Expert B should not rely on them heavily.  
> Note to self: Ensure missing-station artifacts (e.g., Touchet gaps) never leak into gate logic.

For other info see `MDR-TS-v1.0`

> **This answers the question “Does gating help at all?” without changing the core feature pipeline, only the routing logic.**

> Answer: GATING DOES NOT WORK!!!

## 0. Imports

In [117]:
import os
import json
import math
import random
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML / Metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from scipy.special import expit

# Gradient Boosting (baseline model)
from xgboost import XGBRegressor

# PyTorch
import torch

# SciPy
from scipy.special import expit  # sigmoid

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

imports loaded
using: cpu


## 1. Environment Setup

In [118]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

# Environment / Runtime Info
def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    # Colab-specific checks
    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

# Plotting defaults
plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 2.1.2
  Running in Colab: False
  GPU available: False
environment setup complete


## 2. Data Access

In [119]:
# Project paths
VERSION = "v9"
SUBVERSION = "v9.1"
RUN_NAME = "mdr_ts_v9_1"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

# Create output directory if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v9/v9.1

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


## 3. Data Loading

In [120]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_new/train_derived_new.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_new/val_derived_new.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_new/test_derived_new.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

DROP_COLS = ["slope", "elev"]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cols_present = [c for c in DROP_COLS if c in d.columns]
    d.drop(columns=cols_present, inplace=True)
    print(f"{name}: dropped columns {cols_present}")
    print(f"\n{name}: shape={d.shape}")
    print(f"{name}: columns={len(d.columns)}")

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/train_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/val_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/test_derived_new.csv
train: dropped columns ['slope', 'elev']

train: shape=(16972, 354)
train: columns=354
val: dropped columns ['slope', 'elev']

val: shape=(2919, 354)
val: columns=354
test: dropped columns ['slope', 'elev']

test: shape=(2829, 354)
test: columns=354


In [121]:
# Dropping TouchNet because it has no rows in the validation split

DROP_STATION = "Touchet_WA_824"

def drop_station(df, station_id=DROP_STATION):
    before = len(df)
    out = df[df["station_id"] != station_id].copy()
    after = len(out)
    print(f"Dropped {station_id}: {before} -> {after} rows (-{before-after})")
    return out

train_df = drop_station(train_df)
val_df   = drop_station(val_df)
test_df  = drop_station(test_df)

Dropped Touchet_WA_824: 16972 -> 13661 rows (-3311)
Dropped Touchet_WA_824: 2919 -> 2919 rows (-0)
Dropped Touchet_WA_824: 2829 -> 2659 rows (-170)


In [122]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 354

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'aspect', 'DOY', 'soil_moisture_5cm', 'D_sin_DOY', 'D_cos_DOY', 'F_NDVI', 'F_NDMI', 'F_MSI', 'E_SAR_ratio', 'E_SAR_diff', 'G_API', 'G_DSLR', 'G_rain_sum_3d', 'G_rain_sum_7d', 'G_rain_sum_30d', 'A_d_G_API_kobs1', 'A_d_G_API_kobs2', 'A_d_G_API_kobs5']


## 4. Data Sanity Checks

In [123]:
TARGET_COL = "soil_moisture_5cm"
KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

GATE_COLS = [
    "G_rain_sum_7d",
    "G_API",
    "G_DSLR",

    # maybe:
    # "DOY",
    # "LST_modis",
]

# expert A: stable/dry expert
FEATURE_COLS_STABLE = [
    # seasonality
    "DOY",              # maybe: swap to D_sin_DOY/D_cos_DOY
    "D_sa_LST_modis",
    "D_sa_E_SAR_ratio",
    "D_sa_F_NDMI",
    "D_z_E_SAR_ratio",
    "D_z_F_NDMI",

    # SAR/optical state
    "E_SAR_ratio",
    "E_SAR_diff",
    "F_NDMI",
    "F_MSI",
    "s1_vh",
    "s2_b8",
    "s2_b12",

    # lagged/slow memory (non-rain)
    "C_lag_E_SAR_diff_kobs6",
    "C_lag_E_SAR_diff_kobs12",
    "C_lag_E_SAR_diff_kobs30",
    "C_lag_E_SAR_ratio_kobs30",
    "C_lag_F_NDVI_kobs30",
    "C_lag_LST_modis_kobs12",
    "C_lag_LST_modis_kobs30",

    # LST dynamics / volatility
    "A_d_LST_modis_kobs7",
    "A_grad_LST_modis_kobs14",
    "V_ema_LST_modis_kobs30",
    "V_rollmax_LST_modis_kobs7",

    # SAR/optical vol
    "V_rollmax_E_SAR_diff_kobs7",
    "V_rollmax_E_SAR_diff_kobs14",
    "V_rollmax_E_SAR_diff_kobs30",
    "V_rollmax_E_SAR_ratio_kobs14",
    "V_rollmin_E_SAR_diff_kobs30",
    "V_rollmin_E_SAR_ratio_kobs30",
    "V_rollmin_F_NDMI_kobs30",
    "V_rollmax_F_NDVI_kobs14",
    "V_rollmax_F_NDVI_kobs30",
]

# # expert B: dynamic/wet expert
FEATURE_COLS_DYNAMIC = [
    # rain forcing (core)
    "G_rain_sum_3d",
    "G_rain_sum_7d",
    "G_rain_sum_30d",
    "G_API",
    "G_DSLR",
    "V_rollmax_G_API_kobs7",
    "V_rollmax_G_API_kobs30",
    "V_rollmean_G_API_kobs30",
    "V_rollmin_G_API_kobs7",

    # temperature + radar response
    "D_sa_LST_modis",
    "LST_modis",
    "s1_vh",
    "E_SAR_ratio",
    "E_SAR_diff",

    # maybe:
    # "F_NDMI",
    # "s2_b8",
    # "s2_b12",
]

FEATURE_COLS_ALL = sorted(set(FEATURE_COLS_STABLE + FEATURE_COLS_DYNAMIC))

print("Gate cols:", len(GATE_COLS))
print("Stable cols:", len(FEATURE_COLS_STABLE))
print("Dynamic cols:", len(FEATURE_COLS_DYNAMIC))
print("All cols:", len(FEATURE_COLS_ALL))

Gate cols: 3
Stable cols: 33
Dynamic cols: 14
All cols: 43


In [124]:
for d in (train_df, val_df, test_df):
    d["date"] = pd.to_datetime(d["date"], errors="coerce")

stations = sorted(set(train_df["station_id"].dropna().unique())
                  | set(val_df["station_id"].dropna().unique())
                  | set(test_df["station_id"].dropna().unique()))

print("\n=== TEMPORAL LEAKAGE CHECKS (per station) ===")
bad = 0

for sid in stations:
    tr = train_df[train_df["station_id"] == sid]["date"].dropna()
    va = val_df[val_df["station_id"] == sid]["date"].dropna()
    te = test_df[test_df["station_id"] == sid]["date"].dropna()

    if len(tr) == 0 or len(va) == 0 or len(te) == 0:
        print(f"[WARN] station {sid}: missing split data (train={len(tr)}, val={len(va)}, test={len(te)})")
        bad += 1
        continue

    tr_min, tr_max = tr.min(), tr.max()
    va_min, va_max = va.min(), va.max()
    te_min, te_max = te.min(), te.max()

    # date overlap checks (hard leakage)
    overlap_tr_va = len(set(tr.unique()) & set(va.unique()))
    overlap_tr_te = len(set(tr.unique()) & set(te.unique()))
    overlap_va_te = len(set(va.unique()) & set(te.unique()))

    # ordering check (soft but important)
    order_ok = (tr_max < va_min) and (va_max < te_min)

    if overlap_tr_va or overlap_tr_te or overlap_va_te or (not order_ok):
        print(f"[ERROR] station {sid}:")
        print(f"  train: {tr_min} -> {tr_max}")
        print(f"  val:   {va_min} -> {va_max}")
        print(f"  test:  {te_min} -> {te_max}")
        print(f"  overlaps: train∩val={overlap_tr_va}, train∩test={overlap_tr_te}, val∩test={overlap_va_te}")
        print(f"  order_ok: {order_ok}")
        bad += 1
    else:
        print(f"[OK] station {sid}: train<{val_df is not None and 'val' or ''}val<test with no date overlap")

if bad == 0:
    print("\n[INFO] No temporal leakage detected.")
else:
    print(f"\n[WARNING] {bad} station(s) have temporal leakage or split issues.")



=== TEMPORAL LEAKAGE CHECKS (per station) ===
[OK] station Darrington: train<valval<test with no date overlap
[OK] station Quinault: train<valval<test with no date overlap
[OK] station SourdoughGulch_WA_985: train<valval<test with no date overlap
[OK] station Spokane: train<valval<test with no date overlap

[INFO] No temporal leakage detected.


In [125]:
print("\nTemporal split check (train -> validation):")

for sid in sorted(train_df["station_id"].unique()):
    train_dates = train_df.loc[train_df["station_id"] == sid, "date"]
    val_dates   = val_df.loc[val_df["station_id"] == sid, "date"]

    max_train = train_dates.max()
    min_val   = val_dates.min()

    print(f"  Station {sid}:")
    print(f"    train max date: {max_train}")
    print(f"    val   min date: {min_val}")


Temporal split check (train -> validation):
  Station Darrington:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station Quinault:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station SourdoughGulch_WA_985:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station Spokane:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00


## 5. Train / Validation / Test Split

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [126]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     13661
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2011-10-06 00:00:00 -- 2021-09-23 00:00:00

VAL
  rows:     2919
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-09-24 00:00:00 -- 2023-11-12 00:00:00

TEST
  rows:     2659
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2023-11-13 00:00:00 -- 2025-12-31 00:00:00

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


## 6. Model Definition

### 6.1 Feature Matrix Construction

In [127]:
y_train = train_df[TARGET_COL].to_numpy()
y_val   = val_df[TARGET_COL].to_numpy()
y_test  = test_df[TARGET_COL].to_numpy()

X_train_gate = train_df[GATE_COLS].copy()
X_val_gate   = val_df[GATE_COLS].copy()
X_test_gate  = test_df[GATE_COLS].copy()

X_train_stable = train_df[FEATURE_COLS_STABLE].copy()
X_val_stable   = val_df[FEATURE_COLS_STABLE].copy()
X_test_stable  = test_df[FEATURE_COLS_STABLE].copy()

X_train_dynamic = train_df[FEATURE_COLS_DYNAMIC].copy()
X_val_dynamic   = val_df[FEATURE_COLS_DYNAMIC].copy()
X_test_dynamic  = test_df[FEATURE_COLS_DYNAMIC].copy()

meta_train = train_df[KEEP_META_COLS].copy()
meta_val   = val_df[KEEP_META_COLS].copy()
meta_test  = test_df[KEEP_META_COLS].copy()

print("Matrices ready")
print("  Gate:", X_train_gate.shape)
print("  Stable expert:", X_train_stable.shape)
print("  Dynamic expert:", X_train_dynamic.shape)
print("  y_train:", y_train.shape)

Matrices ready
  Gate: (13661, 3)
  Stable expert: (13661, 33)
  Dynamic expert: (13661, 14)
  y_train: (13661,)


In [128]:
GATE_SIGNAL = "G_rain_sum_7d"   # maybe: "G_API", "G_DSLR"
GATE_MODE   = "quantile"        # "quantile" | "fixed"
WET_Q       = 0.75              # only if "quantile"
WET_THR     = 4.0               # only if "fixed"
GATE_K      = 1.0               # soft gate steepness

if GATE_SIGNAL not in train_df.columns:
    raise ValueError(f"GATE_SIGNAL '{GATE_SIGNAL}' not found in dataframe.")

if GATE_MODE == "quantile":
    thr = float(train_df[GATE_SIGNAL].quantile(WET_Q))
elif GATE_MODE == "fixed":
    thr = float(WET_THR)
else:
    raise ValueError(f"Unknown GATE_MODE: {GATE_MODE}")

for d in (train_df, val_df, test_df):
    signal = d[GATE_SIGNAL].astype(float)

    d["is_wet"] = (signal >= thr).astype(int)              # hard gate
    d["g"] = expit(GATE_K * (signal - thr))                # soft gate weight

iswet_train = train_df["is_wet"].to_numpy(dtype=float)
iswet_val   = val_df["is_wet"].to_numpy(dtype=float)
iswet_test  = test_df["is_wet"].to_numpy(dtype=float)

g_train = train_df["g"].to_numpy(dtype=float)
g_val   = val_df["g"].to_numpy(dtype=float)
g_test  = test_df["g"].to_numpy(dtype=float)

print("Gate ready")
print(f"  SIGNAL: {GATE_SIGNAL}")
print(f"  MODE:   {GATE_MODE}")
print(f"  THR:    {thr:.4f}")
print(f"  K:      {GATE_K}")
print("  wet% train/val/test:",
      round(float(iswet_train.mean()), 3),
      round(float(iswet_val.mean()), 3),
      round(float(iswet_test.mean()), 3))

print("  g train mean/min/max:",
      round(float(g_train.mean()), 3),
      round(float(g_train.min()), 3),
      round(float(g_train.max()), 3))

Gate ready
  SIGNAL: G_rain_sum_7d
  MODE:   quantile
  THR:    45.2000
  K:      1.0
  wet% train/val/test: 0.25 0.258 0.256
  g train mean/min/max: 0.251 0.0 1.0


In [129]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

### 6.4.1 Expert A Setup

In [130]:
maskA_train = (train_df["is_wet"].to_numpy() == 0)
maskA_val   = (val_df["is_wet"].to_numpy() == 0)

XA_train = X_train_stable.loc[maskA_train].copy()
yA_train = y_train[maskA_train]

XA_val   = X_val_stable.loc[maskA_val].copy()
yA_val   = y_val[maskA_val]

metaA_train = meta_train.loc[maskA_train].copy()
metaA_val   = meta_val.loc[maskA_val].copy()

print("Expert A (STABLE/DRY) matrices ready")
print("  train rows:", XA_train.shape[0], "wet%:", float(train_df.loc[maskA_train, "is_wet"].mean()))
print("  val rows:  ", XA_val.shape[0],   "wet%:", float(val_df.loc[maskA_val, "is_wet"].mean()))
print("  XA_train:", XA_train.shape, "yA_train:", yA_train.shape)
print("  XA_val:  ", XA_val.shape,   "yA_val:  ", yA_val.shape)

Expert A (STABLE/DRY) matrices ready
  train rows: 10239 wet%: 0.0
  val rows:   2165 wet%: 0.0
  XA_train: (10239, 33) yA_train: (10239,)
  XA_val:   (2165, 33) yA_val:   (2165,)


### 6.4.2 Expert B Setup

In [131]:
maskB_train = (train_df["is_wet"].to_numpy() == 1)
maskB_val   = (val_df["is_wet"].to_numpy() == 1)

XB_train = X_train_dynamic.loc[maskB_train].copy()
yB_train = y_train[maskB_train]

XB_val   = X_val_dynamic.loc[maskB_val].copy()
yB_val   = y_val[maskB_val]

metaB_train = meta_train.loc[maskB_train].copy()
metaB_val   = meta_val.loc[maskB_val].copy()

print("Expert B (WET/DYNAMIC) matrices ready")
print("  train rows:", XB_train.shape[0])
print("  val rows:  ", XB_val.shape[0])
print("  XB_train:", XB_train.shape, "yB_train:", yB_train.shape)
print("  XB_val:  ", XB_val.shape,   "yB_val:  ", yB_val.shape)

Expert B (WET/DYNAMIC) matrices ready
  train rows: 3422
  val rows:   754
  XB_train: (3422, 14) yB_train: (3422,)
  XB_val:   (754, 14) yB_val:   (754,)


### 6.5.1 Expert A (STABLE/DRY) | Model A

In [132]:
xgbA = XGBRegressor(
    subsample=0.9,
    reg_lambda=2.0,
    reg_alpha=0.05,
    n_estimators=4000,
    min_child_weight=3,
    max_depth=7,
    learning_rate=0.05,
    gamma=0.0,
    colsample_bytree=0.75,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
    tree_method="hist",
)

rfA = RandomForestRegressor(
    n_estimators=800,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.5,
    max_depth=16,
    n_jobs=-1,
    random_state=42,
)

In [133]:
# OOF predictions
tscv = TimeSeriesSplit(n_splits=5)

oof_xgb = np.full(len(XA_train), np.nan, dtype=float)
oof_rf  = np.full(len(XA_train), np.nan, dtype=float)

XA_train_np = XA_train.to_numpy()
yA_train_np = np.asarray(yA_train, dtype=float)

for fold, (tr_idx, va_idx) in enumerate(tscv.split(XA_train_np), start=1):
    xgb_f = XGBRegressor(**xgbA.get_params())
    rf_f  = RandomForestRegressor(**rfA.get_params())

    xgb_f.fit(XA_train_np[tr_idx], yA_train_np[tr_idx])
    rf_f.fit(XA_train_np[tr_idx], yA_train_np[tr_idx])

    oof_xgb[va_idx] = xgb_f.predict(XA_train_np[va_idx])
    oof_rf[va_idx]  = rf_f.predict(XA_train_np[va_idx])

mask_oof = np.isfinite(oof_xgb) & np.isfinite(oof_rf)

A_oof = np.vstack([oof_xgb[mask_oof], oof_rf[mask_oof]]).T
y_oof = yA_train_np[mask_oof]

ridgeA = Ridge(alpha=1.0, random_state=42)
ridgeA.fit(A_oof, y_oof)


,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,42


In [134]:
xgbA.fit(XA_train, yA_train)
rfA.fit(XA_train, yA_train)

,n_estimators,800
,criterion,'squared_error'
,max_depth,16
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [135]:
A_pred_val = np.vstack([xgbA.predict(XA_val), rfA.predict(XA_val)]).T
yA_hat_val = ridgeA.predict(A_pred_val)

r2_A   = r2_score(yA_val, yA_hat_val)
mae_A  = mean_absolute_error(yA_val, yA_hat_val)
rmse_A = np.sqrt(mean_squared_error(yA_val, yA_hat_val))

print("Expert A (STABLE/DRY) trained (OOF ridge)")
print("  val metrics (DRY-only):")
print(f"    R2  : {r2_A:.6f}")
print(f"    MAE : {mae_A:.6f}")
print(f"    RMSE: {rmse_A:.6f}")
print("  ridge weights:", ridgeA.coef_.round(6), "intercept:", float(ridgeA.intercept_))

Expert A (STABLE/DRY) trained (OOF ridge)
  val metrics (DRY-only):
    R2  : 0.720849
    MAE : 0.041791
    RMSE: 0.054473
  ridge weights: [0.27778  0.653943] intercept: 0.0058339731978388765


### 6.5.2 Expert B (Model B) | WET

In [136]:
xgbB = XGBRegressor(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=10,
    reg_lambda=5.0,
    subsample=0.8,
    colsample_bytree=0.6,
    gamma=1.0,
    random_state=42,
    n_jobs=-1,
)

rfB = RandomForestRegressor(
    n_estimators=800,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.5,
    max_depth=16,
    n_jobs=-1,
    random_state=42,
)

In [137]:
# OOF predictions
tscv = TimeSeriesSplit(n_splits=5)

oof_xgb = np.full(len(XB_train), np.nan, dtype=float)
oof_rf  = np.full(len(XB_train), np.nan, dtype=float)

XB_train_np = XB_train.to_numpy()
yB_train_np = np.asarray(yB_train, dtype=float)

for fold, (tr_idx, va_idx) in enumerate(tscv.split(XB_train_np), start=1):
    xgb_f = XGBRegressor(**xgbB.get_params())
    rf_f  = RandomForestRegressor(**rfB.get_params())

    xgb_f.fit(XB_train_np[tr_idx], yB_train_np[tr_idx])
    rf_f.fit(XB_train_np[tr_idx], yB_train_np[tr_idx])

    oof_xgb[va_idx] = xgb_f.predict(XB_train_np[va_idx])
    oof_rf[va_idx]  = rf_f.predict(XB_train_np[va_idx])

mask_oof = np.isfinite(oof_xgb) & np.isfinite(oof_rf)

B_oof = np.vstack([oof_xgb[mask_oof], oof_rf[mask_oof]]).T
y_oof = yB_train_np[mask_oof]

ridgeB = Ridge(alpha=1.0, random_state=42)
ridgeB.fit(B_oof, y_oof)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,42


In [138]:
xgbB.fit(XB_train, yB_train)
rfB.fit(XB_train, yB_train)

,n_estimators,800
,criterion,'squared_error'
,max_depth,16
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [139]:
B_pred_val = np.vstack([xgbB.predict(XB_val), rfB.predict(XB_val)]).T
yB_hat_val = ridgeB.predict(B_pred_val)

r2_B   = r2_score(yB_val, yB_hat_val)
mae_B  = mean_absolute_error(yB_val, yB_hat_val)
rmse_B = np.sqrt(mean_squared_error(yB_val, yB_hat_val))

print("Expert B (DYNAMIC/WET) trained (OOF ridge)")
print("  val metrics (WET-only):")
print(f"    R2  : {r2_B:.6f}")
print(f"    MAE : {mae_B:.6f}")
print(f"    RMSE: {rmse_B:.6f}")
print("  ridge weights:", ridgeB.coef_.round(6), "intercept:", float(ridgeB.intercept_))

Expert B (DYNAMIC/WET) trained (OOF ridge)
  val metrics (WET-only):
    R2  : -0.121670
    MAE : 0.036738
    RMSE: 0.046666
  ridge weights: [-0.192196  0.687314] intercept: 0.13528007209928505


## 7. Gating

In [140]:
# expert A predictions on ALL rows
A_pred_train = ridgeA.predict(np.vstack([xgbA.predict(X_train_stable), rfA.predict(X_train_stable)]).T)
A_pred_val   = ridgeA.predict(np.vstack([xgbA.predict(X_val_stable),   rfA.predict(X_val_stable)]).T)
A_pred_test  = ridgeA.predict(np.vstack([xgbA.predict(X_test_stable),  rfA.predict(X_test_stable)]).T)

# expert B predictions on ALL rows
B_pred_train = ridgeB.predict(np.vstack([xgbB.predict(X_train_dynamic), rfB.predict(X_train_dynamic)]).T)
B_pred_val   = ridgeB.predict(np.vstack([xgbB.predict(X_val_dynamic),   rfB.predict(X_val_dynamic)]).T)
B_pred_test  = ridgeB.predict(np.vstack([xgbB.predict(X_test_dynamic),  rfB.predict(X_test_dynamic)]).T)

print("Expert predictions ready")
print("  A train/val/test:", A_pred_train.shape, A_pred_val.shape, A_pred_test.shape)
print("  B train/val/test:", B_pred_train.shape, B_pred_val.shape, B_pred_test.shape)

Expert predictions ready
  A train/val/test: (13661,) (2919,) (2659,)
  B train/val/test: (13661,) (2919,) (2659,)


In [141]:
iswet_train = train_df["is_wet"].to_numpy().astype(float)
iswet_val   = val_df["is_wet"].to_numpy().astype(float)
iswet_test  = test_df["is_wet"].to_numpy().astype(float)

yhat_train_hard = iswet_train * B_pred_train + (1 - iswet_train) * A_pred_train
yhat_val_hard   = iswet_val   * B_pred_val   + (1 - iswet_val)   * A_pred_val
yhat_test_hard  = iswet_test  * B_pred_test  + (1 - iswet_test)  * A_pred_test

In [142]:
g_train = train_df["g"].to_numpy().astype(float)
g_val   = val_df["g"].to_numpy().astype(float)
g_test  = test_df["g"].to_numpy().astype(float)

yhat_train_soft = g_train * B_pred_train + (1 - g_train) * A_pred_train
yhat_val_soft   = g_val   * B_pred_val   + (1 - g_val)   * A_pred_val
yhat_test_soft  = g_test  * B_pred_test  + (1 - g_test)  * A_pred_test

In [143]:
def report(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:>15} | R2={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}")
    return r2, mae, rmse

In [144]:
print("\nHard gate metrics:")
report("train (hard)", y_train, yhat_train_hard)
report("val (hard)",   y_val,   yhat_val_hard)
report("test (hard)",  y_test,  yhat_test_hard)

print("\nSoft gate metrics:")
report("train (soft)", y_train, yhat_train_soft)
report("val (soft)",   y_val,   yhat_val_soft)
report("test (soft)",  y_test,  yhat_test_soft)


Hard gate metrics:
   train (hard) | R2=0.9604  MAE=0.0152  RMSE=0.0204
     val (hard) | R2=0.7276  MAE=0.0405  RMSE=0.0526
    test (hard) | R2=0.6814  MAE=0.0432  RMSE=0.0542

Soft gate metrics:
   train (soft) | R2=0.9609  MAE=0.0152  RMSE=0.0203
     val (soft) | R2=0.7285  MAE=0.0405  RMSE=0.0525
    test (soft) | R2=0.6822  MAE=0.0431  RMSE=0.0542


(0.6821879559801618, 0.04310817114125632, 0.054156280004622394)

## 8. Diagnostics & Evaluation of Approach

In [145]:
print("\nExpert A alone:")
report("val (A)", y_val, A_pred_val)
report("test (A)", y_test, A_pred_test)

print("\nExpert B alone:")
report("val (B)", y_val, B_pred_val)
report("test (B)", y_test, B_pred_test)


Expert A alone:
        val (A) | R2=0.6267  MAE=0.0484  RMSE=0.0615
       test (A) | R2=0.5447  MAE=0.0526  RMSE=0.0648

Expert B alone:
        val (B) | R2=0.1204  MAE=0.0727  RMSE=0.0945
       test (B) | R2=0.0168  MAE=0.0706  RMSE=0.0953


(0.016753153927295705, 0.07055218663259467, 0.09525656800202952)

In [146]:
mask_val_wet = val_df["is_wet"].to_numpy() == 1
mask_val_dry = ~mask_val_wet

print("\nval regime metrics:")
report("val dry (A)", y_val[mask_val_dry], A_pred_val[mask_val_dry])
report("val dry (B)", y_val[mask_val_dry], B_pred_val[mask_val_dry])
report("val wet (A)", y_val[mask_val_wet], A_pred_val[mask_val_wet])
report("val wet (B)", y_val[mask_val_wet], B_pred_val[mask_val_wet])


val regime metrics:
    val dry (A) | R2=0.7208  MAE=0.0418  RMSE=0.0545
    val dry (B) | R2=-0.0604  MAE=0.0853  RMSE=0.1062
    val wet (A) | R2=-2.1620  MAE=0.0675  RMSE=0.0784
    val wet (B) | R2=-0.1217  MAE=0.0367  RMSE=0.0467


(-0.12167026637238254, 0.03673795240078637, 0.04666639280659795)

---

_Jakob Balkovec_